# Daily News Intelligence Agent — Ribbon Communications
## MVP Prototype

This notebook implements a minimum viable prototype of a daily news-intelligence agent focused on **Ribbon Communications**.

**Objective:**
- Fetch yesterday's news from curated business, financial, and telecom sources
- Score relevance to Ribbon's business profile (Cloud & Edge, IP Optical segments)
- Enrich with verified context and business impact estimates
- Generate a concise daily digest
- Store raw and processed data locally

**Scope:**
This is a **validation prototype**, not production automation. It demonstrates end-to-end workflow on a single company profile.

## What This MVP Does

✓ Fetches article metadata from news APIs for the previous day  
✓ Extracts readable article text from accessible URLs  
✓ Scores relevance to Ribbon based on transparent heuristics  
✓ Enriches selected articles with verified context  
✓ Estimates business impact (positive/negative/neutral/mixed)  
✓ Stores results locally in CSV and JSON  
✓ Generates a human-readable daily digest  

## What This MVP Does NOT Do

✗ Build a multi-company generic system  
✗ Include scheduling or daemon infrastructure  
✗ Use browser automation or paid paywalls  
✗ Provide perfect relevance or impact scoring  
✗ Include a production database or API server  
✗ Handle historical trend analysis (optional stub provided)  

**This notebook is designed to be run once per day on a laptop and extended later into a full agent.**

## Section 1: Setup and Configuration

Install required packages, import libraries, configure logging, and initialize the environment.

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'requests',
    'pandas',
    'python-dotenv',
    'trafilatura',
    'python-dateutil',
    'tqdm'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All required packages installed.")

In [ ]:
import os
import json
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import requests
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
import trafilatura
from dateutil.parser import parse as parse_date

# Load environment variables
load_dotenv()

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Create output directories
for dir_path in ['data/raw', 'data/processed', 'outputs']:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

print("✓ Environment and logging configured.")

## Section 2: Define Company Profile and Sources

Define Ribbon Communications profile, curated source allowlist, and helper functions.

In [ ]:
# RIBBON COMMUNICATIONS PROFILE
COMPANY_PROFILE = {
    'company_name': 'Ribbon Communications',
    'company_aliases': [
        'Ribbon',
        'Ribbon Comms',
        'Ribbon Corp',
        'RBBN',
        'Ribbon Communications Inc',
    ],
    'sector': 'Telecommunications',
    'segments': {
        'Cloud & Edge': {
            'description': 'Cloud and edge computing infrastructure and services',
            'keywords': ['cloud', 'edge', 'RAN', 'vRAN', 'containerized', 'NFV'],
            'customers': ['operators', 'enterprises', 'government', 'defense'],
        },
        'IP Optical': {
            'description': 'IP and optical networking solutions',
            'keywords': ['IP optical', 'routing', 'switching', 'optical', 'bandwidth'],
            'customers': ['mobile operators', 'data centers', 'cloud providers', 'government'],
        },
    },
    'key_customers': [
        'AT&T', 'Verizon', 'Deutsche Telekom', 'Vodafone', 'Orange',
        'BT', 'Telefonica', 'Swisscom', 'Telenor', 'SoftBank',
        'NTT', 'China Mobile', 'India Jio', 'Amazon', 'Microsoft', 'Google'
    ],
    'geographies': ['US', 'EMEA', 'Asia Pacific'],
    'distribution': {
        'direct': 'Direct sales to large operators and cloud providers',
        'indirect': 'System integrators, distributors/resellers, operator partners',
        'model': 'Hybrid, direct-heavy'
    },
}

print(f"✓ Loaded company profile: {COMPANY_PROFILE['company_name']}")
print(f"  Segments: {', '.join(COMPANY_PROFILE['segments'].keys())}")
print(f"  Geographies: {', '.join(COMPANY_PROFILE['geographies'])}")

In [ ]:
# Curated allowlist of reputable business, financial, telecom, and regulatory sources
CURATED_SOURCES = {
    'reuters.com': 'Reuters',
    'bloomberg.com': 'Bloomberg',
    'ft.com': 'Financial Times',
    'wsj.com': 'Wall Street Journal',
    'cnbc.com': 'CNBC',
    'businessinsider.com': 'Business Insider',
    'thestreet.com': 'TheStreet',
    'rcrwireless.com': 'RCR Wireless News',
    'fiercewireless.com': 'Fierce Wireless',
    'fiercetelecom.com': 'Fierce Telecom',
    'ispreview.co.uk': 'ISP Review',
    'lightreading.com': 'Light Reading',
    'telecompaper.com': 'Telecompaper',
    'eetimes.com': 'EE Times',
    'networkworld.com': 'Network World',
    'sdxcentral.com': 'SDx Central',
    'techcrunch.com': 'TechCrunch',
    'theverge.com': 'The Verge',
    'arstechnica.com': 'Ars Technica',
    'theregister.com': 'The Register',
    'venturebeat.com': 'VentureBeat',
    'investors.ribbon.com': 'Ribbon Investor Relations',
    'ribbon.com': 'Ribbon Communications',
    'sec.gov': 'SEC EDGAR',
    'prnewswire.com': 'PR Newswire',
    'businesswire.com': 'Business Wire',
    'fcc.gov': 'FCC',
    'ofcom.org.uk': 'Ofcom',
    'ec.europa.eu': 'European Commission',
}

def get_source_from_url(url: str) -> Optional[str]:
    """Extract domain and map to source name."""
    try:
        from urllib.parse import urlparse
        domain = urlparse(url).netloc.lower().replace('www.', '')
        return CURATED_SOURCES.get(domain, None)
    except:
        return None

print(f"✓ Loaded {len(CURATED_SOURCES)} trusted sources")

## Section 3: Build Search Strategy

Create search queries and configure API keys for article discovery.

In [ ]:
def build_search_queries() -> List[str]:
    """Build a list of search queries for news discovery."""
    queries = []
    
    # 1. Direct company mentions
    queries.extend([
        'Ribbon Communications',
        'Ribbon telecom',
        'RBBN stock',
    ])
    
    # 2. Segment-specific: Cloud & Edge
    queries.extend([
        'telecom cloud RAN vRAN',
        'network edge computing telco',
        'operator cloud infrastructure',
        'virtual RAN vRAN deployment',
    ])
    
    # 3. Segment-specific: IP Optical
    queries.extend([
        'IP optical networking telecom',
        'optical routing switching infrastructure',
        'data center interconnect',
    ])
    
    # 4. Key customers
    queries.extend([
        'AT&T cloud network infrastructure',
        'Verizon 5G cloud RAN',
        'Deutsche Telekom network transformation',
        'Vodafone OpenRAN 5G',
        'Orange network modernization',
    ])
    
    # 5. Market drivers and competitive signals
    queries.extend([
        'telecom equipment vendor consolidation',
        'NFV network function virtualization',
        'OpenRAN ecosystem developments',
        'telecom infrastructure investment',
    ])
    
    # 6. Regulatory and macro signals
    queries.extend([
        'telecom regulatory reform',
        'US telecom infrastructure bill',
    ])
    
    return list(set(queries))  # Remove duplicates

search_queries = build_search_queries()
print(f"✓ Built {len(search_queries)} search queries")
print(f"  Sample: {search_queries[:5]}")

In [ ]:
# NewsAPI key for fetching latest news
NEWS_API_KEY = 'f63e6476-13e8-44c2-a50e-8508b4ad68d6'
print("✓ NEWS_API_KEY configured for latest news")

# Date configuration
ANALYSIS_DATE = datetime.utcnow().date()
SEARCH_FROM_DATE = (ANALYSIS_DATE - timedelta(days=1)).isoformat()  # Yesterday
SEARCH_TO_DATE = ANALYSIS_DATE.isoformat()  # Today midnight

print(f"✓ Analysis date: {ANALYSIS_DATE}")
print(f"  Searching for articles from: {SEARCH_FROM_DATE} to {SEARCH_TO_DATE}")

## Section 4: Fetch and Extract Articles

Fetch article metadata from NewsAPI and extract readable content using trafilatura.

In [ ]:
def fetch_articles_from_newsapi(queries: List[str], from_date: str, to_date: str) -> List[Dict]:
    """Fetch articles from NewsAPI for a list of queries."""
    import time
    
    if not NEWS_API_KEY or NEWS_API_KEY == 'demo':
        logger.error("NewsAPI key not configured. Please set a valid API key.")
        return []
    
    articles = []
    seen_urls = set()
    api_url = 'https://newsapi.org/v2/everything'
    first_request = True
    
    for query in tqdm(queries, desc='Fetching articles'):
        try:
            # Add delay between requests to respect rate limits
            if not first_request:
                time.sleep(0.5)
            first_request = False
            
            params = {
                'q': query,
                'from': from_date,
                'to': to_date,
                'sortBy': 'relevancy',
                'language': 'en',
                'apiKey': NEWS_API_KEY,
            }
            
            response = requests.get(api_url, params=params, timeout=10)
            
            if response.status_code == 200:
                elif data.get('status') == 'error':
                    logger.warning(f"API error for query '{query}': {data.get('message')}")
            elif response.status_code == 429:
                logger.warning(f"Rate limited for query '{query}'. Pausing...")
                time.sleep(2)
            else:
                logger.warning(f"API error for query '{query}': {response.status_code}")
                
        except requests.exceptions.RequestException as e:
            logger.warning(f"Request error for query '{query}': {str(e)}")
        except Exception as e:
            logger.warning(f"Error processing query '{query}': {str(e)}")
    
    return articles
if not raw_articles:
    print("\n⚠ WARNING: No articles fetched. Check API key and network connectivity.")
else:

    print(f"\n✓ Fetched {len(raw_articles)} articles from curated sources")
raw_articles = fetch_articles_from_newsapi(search_queries, SEARCH_FROM_DATE, SEARCH_TO_DATE)
            logger.warning(f"Error processing query '{query}': {str(e)}")    print(f"  Sample sources: {set([a.get('source_name') for a in raw_articles[:5]])}")

    print(f"  Sample sources: {set([a.get('source_name') for a in raw_articles[:5]])}")

    if raw_articles:

print(f"\n✓ Fetched {len(raw_articles)} articles from curated sources")
    return articlesprint(f"\n✓ Fetched {len(raw_articles)} articles from curated sources")

if raw_articles:


    print(f"  Sample sources: {set([a.get('source_name') for a in raw_articles[:5]])}")
# Fetch articlesraw_articles = fetch_articles_from_newsapi(search_queries, SEARCH_FROM_DATE, SEARCH_TO_DATE)
print("Fetching articles from NewsAPI...")

In [ ]:
def extract_article_content(url: str, timeout: int = 10) -> Tuple[Optional[str], str]:
    """Extract article text from URL using trafilatura."""
    try:
        response = requests.get(url, timeout=timeout, headers={
            'User-Agent': 'Mozilla/5.0 (compatible; NewsAgent/1.0)'
        })
        response.raise_for_status()
        
        # Try trafilatura extraction
        text = trafilatura.extract(response.text, include_comments=False)
        
        if text:
            return text[:5000], 'extracted'  # Truncate to 5000 chars
        else:
            return None, 'no_text_found'
            
    except requests.exceptions.Timeout:
        return None, 'timeout'
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            return None, 'forbidden'
        elif e.response.status_code == 404:
            return None, 'not_found'
        else:
            return None, f'http_{e.response.status_code}'
    except requests.exceptions.RequestException as e:
        return None, 'request_error'
    except Exception as e:
        return None, 'extraction_error'

# Extract content for each article
print("Extracting article content...")
for article in tqdm(raw_articles, desc='Extracting'):
    url = article.get('url')
    if url:
        content, status = extract_article_content(url)
        article['article_text'] = content
        article['extraction_status'] = status
    else:
        article['article_text'] = None
        article['extraction_status'] = 'no_url'

extracted_count = sum(1 for a in raw_articles if a.get('extraction_status') == 'extracted')
print(f"\n✓ Extracted content from {extracted_count}/{len(raw_articles)} articles")

## Section 5: Normalize and Score Relevance

Create a clean DataFrame and score relevance to Ribbon's business profile.

In [ ]:
def normalize_articles(articles: List[Dict]) -> pd.DataFrame:
    """Normalize raw article data into a clean DataFrame."""
    records = []
    
    for i, article in enumerate(articles):
        pub_date = article.get('publishedAt')
        try:
            if pub_date:
                pub_date = parse_date(pub_date).isoformat()
        except:
            pub_date = None
        
        record = {
            'analysis_date': ANALYSIS_DATE.isoformat(),
            'article_id': f"article_{i}_{hash(article.get('url', '')) % 10000}",
            'title': article.get('title', ''),
            'description': article.get('description', ''),
            'source_name': article.get('source_name', 'Unknown'),
            'source_domain': article.get('source_domain', ''),
            'published_at': pub_date,
            'url': article.get('url', ''),
            'extraction_status': article.get('extraction_status', 'unknown'),
            'article_text': article.get('article_text', ''),
            'author': article.get('author', ''),
            'image_url': article.get('urlToImage', ''),
            'matched_company': None,
            'matched_segments': [],
            'matched_customers': [],
            'matched_geographies': [],
            'relevance_score': 0.0,
            'relevance_reason': '',
            'impact_label': 'neutral',
            'impact_score': 0,
            'impact_confidence': 'low',
            'enrichment_summary': '',
            'kept_for_digest': False,
        }
        records.append(record)
    
    df = pd.DataFrame(records)
    return df

df_articles = normalize_articles(raw_articles)
print(f"✓ Normalized {len(df_articles)} articles into DataFrame")
print(f"\nDataFrame shape: {df_articles.shape}")
print(f"Columns: {list(df_articles.columns)}")

In [ ]:
def score_relevance(row: pd.Series, company_profile: Dict) -> Tuple[float, str, Dict]:
    """Score relevance of an article to the company profile."""
    score = 0.0
    matches = {
        'company': [],
        'segments': [],
        'customers': [],
        'geographies': [],
        'keywords': [],
    }
    
    # Combine searchable text
    text = ' '.join([
        str(row.get('title', '')),
        str(row.get('description', '')),
        str(row.get('article_text', ''))[:1000]
    ]).lower()
    
    reasons = []
    
    # 1. Direct company mentions
    for alias in company_profile['company_aliases']:
        if alias.lower() in text:
            score += 0.4
            matches['company'].append(alias)
            reasons.append(f"Direct mention of {alias}")
            break
    
    # 2. Segment and product relevance
    for segment_name, segment_info in company_profile['segments'].items():
        for keyword in segment_info['keywords']:
            if keyword.lower() in text:
                score += 0.15
                matches['segments'].append(segment_name)
                reasons.append(f"{segment_name} keyword: {keyword}")
                break
    
    # 3. Key customer mentions
    for customer in company_profile['key_customers']:
        if customer.lower() in text:
            score += 0.10
            matches['customers'].append(customer)
            reasons.append(f"Key customer: {customer}")
    
    # 4. Geography matches
    geo_patterns = {'us': 'US', 'emea': 'EMEA', 'europe': 'EMEA', 'asia': 'Asia Pacific', 'apac': 'Asia Pacific'}
    for pattern, geo in geo_patterns.items():
        if pattern in text and geo not in matches['geographies']:
            score += 0.05
            matches['geographies'].append(geo)
            reasons.append(f"Geography: {geo}")
    
    # 5. Distribution and market signals
    distribution_keywords = ['system integrator', 'distributor', 'reseller', 'partner']
    for kw in distribution_keywords:
        if kw in text:
            score += 0.05
            reasons.append(f"Distribution signal: {kw}")
            break
    
    # 6. Competitive and market signals
    competitive_keywords = ['openran', 'open ran', 'nfv', 'vran', 'telecom equipment']
    for kw in competitive_keywords:
        if kw in text:
            score += 0.08
            matches['keywords'].append(kw)
            reasons.append(f"Market signal: {kw}")
    
    score = min(score, 1.0)
    reason = ' | '.join(reasons) if reasons else 'No direct relevance signals found'
    
    return score, reason, matches

# Score all articles
print("Scoring relevance to Ribbon...")
for idx, row in df_articles.iterrows():
    score, reason, matches = score_relevance(row, COMPANY_PROFILE)
    df_articles.at[idx, 'relevance_score'] = score
    df_articles.at[idx, 'relevance_reason'] = reason
    df_articles.at[idx, 'matched_company'] = matches['company'][0] if matches['company'] else None
    df_articles.at[idx, 'matched_segments'] = matches['segments']
    df_articles.at[idx, 'matched_customers'] = matches['customers']
    df_articles.at[idx, 'matched_geographies'] = matches['geographies']

print(f"\n✓ Relevance scoring complete")
print(f"\nRelevance score distribution:")
print(df_articles['relevance_score'].describe())
print(f"\nArticles with relevance > 0.3: {(df_articles['relevance_score'] > 0.3).sum()}")

## Section 6: Estimate Business Impact

Filter articles, enrich them with context, and score business impact.

In [ ]:
def create_enrichment_summary(row: pd.Series) -> str:
    """Create a human-readable enrichment summary."""
    summary_parts = []
    
    if row.get('matched_segments'):
        summary_parts.append(f"Affects: {', '.join(row['matched_segments'])}")
    
    if row.get('matched_customers'):
        summary_parts.append(f"Key customers: {', '.join(row['matched_customers'][:3])}")
    
    if row.get('matched_geographies'):
        summary_parts.append(f"Geographies: {', '.join(row['matched_geographies'])}")
    
    return ' | '.join(summary_parts) if summary_parts else 'Related to telecom infrastructure'

# Apply enrichment
print("Enriching high-relevance articles...")
for idx, row in df_articles.iterrows():
    df_articles.at[idx, 'enrichment_summary'] = create_enrichment_summary(row)

# Filter for digest: articles with relevance > 0.2 and successful extraction
df_filtered = df_articles[
    (df_articles['relevance_score'] > 0.2) &
    (df_articles['extraction_status'] == 'extracted')
].copy()

print(f"\n✓ Filtered to {len(df_filtered)} articles for further analysis")

In [ ]:
def score_business_impact(row: pd.Series, company_profile: Dict) -> Tuple[str, int, str]:
    """Estimate business impact on Ribbon."""
    text = ' '.join([
        str(row.get('title', '')),
        str(row.get('description', '')),
        str(row.get('article_text', ''))[:1500]
    ]).lower()
    
    score = 0
    confidence = 'low'
    
    # Positive signals
    positive_keywords = [
        'investment', 'acquisition', 'partnership', 'growth', 'expansion',
        'contract', 'deployment', 'selection', 'mandate', 'adoption',
        'increase', 'demand', 'innovation', 'leadership', 'success'
    ]
    
    # Negative signals
    negative_keywords = [
        'decline', 'loss', 'reduction', 'layoff', 'bankruptcy',
        'investigation', 'fine', 'penalty', 'risk', 'threat',
        'competition', 'price pressure', 'margin', 'warning'
    ]
    
    positive_count = sum(1 for kw in positive_keywords if kw in text)
    negative_count = sum(1 for kw in negative_keywords if kw in text)
    
    # Score calculation
    if positive_count > 0 and negative_count == 0:
        score = min(positive_count, 2)
        confidence = 'high' if positive_count > 1 else 'medium'
    elif negative_count > 0 and positive_count == 0:
        score = max(-negative_count, -2)
        confidence = 'high' if negative_count > 1 else 'medium'
    elif positive_count > 0 and negative_count > 0:
        score = 0
        confidence = 'medium'
    else:
        score = 0
        confidence = 'low'
    
    # Impact label
    if score > 0:
        impact_label = 'positive'
    elif score < 0:
        impact_label = 'negative'
    elif positive_count > 0 and negative_count > 0:
        impact_label = 'mixed'
    else:
        impact_label = 'neutral'
    
    return impact_label, score, confidence

# Score impact for filtered articles
print("Scoring business impact...")
for idx in df_filtered.index:
    impact_label, impact_score, confidence = score_business_impact(df_articles.loc[idx], COMPANY_PROFILE)
    df_articles.at[idx, 'impact_label'] = impact_label
    df_articles.at[idx, 'impact_score'] = impact_score
    df_articles.at[idx, 'impact_confidence'] = confidence

# Mark high-confidence articles for digest
digest_threshold = 0.3
df_articles['kept_for_digest'] = (
    (df_articles['relevance_score'] > digest_threshold) &
    (df_articles['extraction_status'] == 'extracted') &
    (df_articles['impact_confidence'].isin(['medium', 'high']))
)

digest_count = df_articles['kept_for_digest'].sum()
print(f"\n✓ Impact scoring complete")
print(f"  Articles marked for digest: {digest_count}")
print(f"\nImpact label distribution (all articles):")
print(df_articles['impact_label'].value_counts())

## Section 7: Generate and Save Outputs

Save raw and filtered articles to CSV and export digest data to JSON.

In [ ]:
# Convert list columns to strings for CSV export
df_export = df_articles.copy()
for col in ['matched_segments', 'matched_customers', 'matched_geographies']:
    df_export[col] = df_export[col].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')

# Save raw articles (all fetched)
raw_csv_path = f'data/raw/articles_raw_{ANALYSIS_DATE.isoformat()}.csv'
df_export.to_csv(raw_csv_path, index=False)
print(f"✓ Saved raw articles: {raw_csv_path}")

# Save filtered articles (high-relevance only)
df_filtered_export = df_export[df_export['relevance_score'] > 0.2].copy()
filtered_csv_path = f'data/processed/articles_filtered_{ANALYSIS_DATE.isoformat()}.csv'
df_filtered_export.to_csv(filtered_csv_path, index=False)
print(f"✓ Saved filtered articles: {filtered_csv_path}")

# Save digest articles as JSON for easier formatting
digest_articles = df_articles[df_articles['kept_for_digest']].copy()
digest_json = []
for _, row in digest_articles.iterrows():
    digest_json.append({
        'title': row['title'],
        'source': row['source_name'],
        'published_at': row['published_at'],
        'url': row['url'],
        'relevance_score': float(row['relevance_score']),
        'impact_label': row['impact_label'],
        'impact_score': int(row['impact_score']),
        'matched_segments': row['matched_segments'] if isinstance(row['matched_segments'], list) else [],
        'matched_customers': row['matched_customers'] if isinstance(row['matched_customers'], list) else [],
        'enrichment_summary': row['enrichment_summary'],
    })

digest_json_path = f'data/processed/digest_articles_{ANALYSIS_DATE.isoformat()}.json'
with open(digest_json_path, 'w') as f:
    json.dump(digest_json, f, indent=2)
print(f"✓ Saved digest articles JSON: {digest_json_path}")

print(f"\n✓ All data saved to data/ directories")

## Section 8: Display Daily Digest

Generate and display a human-readable daily intelligence brief.

In [ ]:
def generate_daily_digest(df, company_profile, analysis_date):
    """
    Generate a human-readable daily intelligence brief.
    
    Parameters:
    - df: DataFrame of articles with relevance and impact scores
    - company_profile: Company profile dict with name and metadata
    - analysis_date: Date string for the digest
    """
    # Filter articles kept for digest
    digest_articles = df[df['kept_for_digest'] == True].copy()
    
    if len(digest_articles) == 0:
        print("No articles met the criteria for today's digest.")
        return
    
    # Sort by relevance score descending
    digest_articles = digest_articles.sort_values('relevance_score', ascending=False)
    
    # Count impact labels
    impact_counts = digest_articles['impact_label'].value_counts().to_dict()
    
    # Build digest output
    digest = []
    digest.append("=" * 80)
    digest.append(f"DAILY NEWS INTELLIGENCE DIGEST - {company_profile['company_name']}")
    digest.append(f"Analysis Date: {analysis_date}")
    digest.append("=" * 80)
    digest.append("")
    
    # Executive summary
    digest.append("EXECUTIVE SUMMARY")
    digest.append("-" * 80)
    digest.append(f"Total Articles Analyzed: {len(df)}")
    digest.append(f"Articles in Digest: {len(digest_articles)}")
    digest.append(f"Positive Impact: {impact_counts.get('positive', 0)}")
    digest.append(f"Negative Impact: {impact_counts.get('negative', 0)}")
    digest.append(f"Neutral: {impact_counts.get('neutral', 0)}")
    digest.append(f"Mixed: {impact_counts.get('mixed', 0)}")
    digest.append("")
    
    # Detailed articles
    digest.append("FEATURED ARTICLES")
    digest.append("-" * 80)
    
    for idx, (_, article) in enumerate(digest_articles.iterrows(), 1):
        digest.append("")
        digest.append(f"[{idx}] {article['title']}")
        digest.append(f"    Source: {article['source_name']}")
        digest.append(f"    Published: {article['published_at']}")
        digest.append(f"    Relevance Score: {article['relevance_score']:.2f}")
        digest.append(f"    Impact: {article['impact_label'].upper()} (confidence: {article['impact_confidence']})")
        if pd.notna(article['matched_segments']) and article['matched_segments']:
            segments_str = ', '.join(article['matched_segments']) if isinstance(article['matched_segments'], list) else str(article['matched_segments'])
            digest.append(f"    Segments: {segments_str}")
        if pd.notna(article['matched_customers']) and article['matched_customers']:
            customers_str = ', '.join(article['matched_customers']) if isinstance(article['matched_customers'], list) else str(article['matched_customers'])
            digest.append(f"    Customers: {customers_str}")
        if pd.notna(article['enrichment_summary']) and article['enrichment_summary']:
            digest.append(f"    Context: {article['enrichment_summary']}")
        digest.append(f"    URL: {article['url']}")
    
    digest.append("")
    digest.append("=" * 80)
    
    # Print to console
    digest_text = "\n".join(digest)
    print(digest_text)
    
    # Save to file
    digest_filename = f"outputs/daily_digest_{analysis_date}.txt"
    with open(digest_filename, 'w') as f:
        f.write(digest_text)
    
    print(f"\nDigest saved to: {digest_filename}")


# Generate and display the daily digestgenerate_daily_digest(df_articles, COMPANY_PROFILE, ANALYSIS_DATE)